# 从文档到可引用答案：在 CANNLab 构建本地 RAG 知识库

本 Notebook 面向希望在昇腾环境中完成一次端到端 AI 应用开发的开发者。你将从一份真实文档开始，依次完成文档处理、向量检索、本地 NPU 模型部署、证据增强问答和结果验证，而不是只运行一个孤立的模型推理示例。

RAG（检索增强生成）的价值在于：模型回答问题前，先从指定知识库中检索证据，再依据证据生成答案。这样可以让回答与私有文档建立可追溯关系，同时减少依赖模型内部记忆造成的事实偏差。

> 本文代码已经在 CANNLab A3 环境完成运行验证。所有文档处理、向量化、索引和生成均在当前实例内完成，不调用外部在线大模型 API；首次运行仍需访问 GitCode 和 ModelScope 下载示例文档及模型。

### 建议读者

- 了解基本 Python 和 Notebook 操作；
- 希望理解一个最小 RAG 应用各模块之间的关系；
- 需要在 CANNLab 实例内处理文档并使用本地 NPU 模型生成答案。

### 执行方式

首次体验请从上到下依次运行全部代码单元，不要跳过配置、文档处理或服务验证步骤。模型权重合计约 20 GB，实际耗时主要取决于网络下载速度和模型加载速度。

## 0. 先明确成果与边界

完成后，可以获得以下文件与运行结果：

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">产物</th><th align="left">用途</th></tr>
<tr><td align="left">CANNLab 用户手册及图片</td><td align="left">示例知识来源</td></tr>
<tr><td align="left">FAISS 向量索引</td><td align="left">保存文档块向量，支持相似度检索</td></tr>
<tr><td align="left">JSONL 元数据</td><td align="left">保存来源文件、章节、行号、正文和图片引用</td></tr>
<tr><td align="left">bge-m3 模型</td><td align="left">为文档和问题生成可比较的向量</td></tr>
<tr><td align="left">Qwen3-8B 本地服务</td><td align="left">运行于本地进程，根据检索证据生成回答</td></tr>
<tr><td align="left">冒烟评测表</td><td align="left">在 Notebook 输出区展示，供人工核查回答、拒答、相似度和耗时</td></tr>
</table>

文档、模型、向量索引和元数据保存在工作目录中；评测表不会自动导出为 CSV 或 JSON 文件。如需保留本次评测记录，请运行评测单元后保存包含输出的 Notebook。

最终应用链路为：

> 问题 → 向量化 → Top-K 检索 → 证据组装 → 本地模型生成 → 引用与相关图片展示

### 当前范围

- 解析 Markdown 和 TXT 文档；
- 记录本地 Markdown 图片引用，但不执行图片 OCR 或多模态理解；
- 使用 bge-m3、FAISS 和余弦相似度完成基础召回；
- 使用本地 Qwen3-8B，根据证据回答或拒答；
- 提供文件名、章节、行号和检索分数，便于人工追溯。

### 非目标

本版本不包含 PDF 解析、OCR、重排序、图形界面、多用户并发、身份认证和正式评测平台。HTTP 服务只实现本实践需要的 `GET /v1/models` 与 `POST /v1/chat/completions` 核心字段，不等同于完整 OpenAI API，也不适合作为未经加固的生产服务直接对外开放。

## 1. 看懂这条 RAG 链路

```text
离线建库阶段
示例文档 → 标题感知切分 → bge-m3 文档向量化 → FAISS 索引
                │                                  │
                └── 来源、章节、行号、图片 ──→ JSONL 元数据

在线问答阶段
用户问题 → bge-m3 问题向量化 → Top-K 检索 → 相似度门槛
                                                │
                         ┌───────────────不足────┴────足够──────────────┐
                         ↓                                             ↓
                       拒答                              证据与问题组装成提示词
                                                                       ↓
                                                         本地 Qwen3-8B（NPU）
                                                                       ↓
                                                            回答、引用、相关图片
```

这里有两个需要区分的阶段：

1. **建库**：文档批量向量化后写入索引，通常只在文档变化时重新执行；
2. **问答**：每个问题只生成一个问题向量，再从已有索引中检索证据并调用生成模型。

FAISS 使用内积索引，但文档向量和问题向量都经过归一化，因此这里的内积分数等价于余弦相似度。相关图片仅来自命中文档块的路径信息，模型本身没有读取或理解图片内容。

## 2. 准备 CANNLab 环境

本实践已在 CANNLab A3 环境的 `py3.12-A3-arm-vllm-20260928` 镜像完成验证，代码使用映射后的逻辑设备 `npu:0`。开始前请确认：

- NPU 可用且健康；
- `/mnt/workspace` 至少有 35 GB 可用空间；
- 当前实例可访问 GitCode 和 ModelScope；
- 本机端口 `8765` 未被其他服务占用。

35 GB 是考虑模型文件、下载过程、索引和日志后的最低检查值，并非所有环境下的精确占用量。

### 已验证的软件基线

以下版本来自 2026-09-29 的实际验证环境，用于说明已知可工作的组合，不代表最低兼容版本：

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">组件</th><th align="left">已验证版本</th></tr>
<tr><td align="left">CANNLab 镜像</td><td align="left"><code>py3.12-A3-arm-vllm-20260928</code></td></tr>
<tr><td align="left">操作系统 / 架构</td><td align="left">Ubuntu 24.04.3 LTS / aarch64</td></tr>
<tr><td align="left">NPU</td><td align="left">1 张 Ascend 910C（2 个芯片、128 GiB HBM）</td></tr>
<tr><td align="left">驱动 / npu-smi</td><td align="left">25.5.5 / 25.5.5</td></tr>
<tr><td align="left">CANN</td><td align="left">9.1.0</td></tr>
<tr><td align="left">Python</td><td align="left">3.12.9</td></tr>
<tr><td align="left">PyTorch</td><td align="left">2.10.0+cpu</td></tr>
<tr><td align="left">torch-npu</td><td align="left">2.10.0.post4</td></tr>
<tr><td align="left">Transformers</td><td align="left">5.5.4</td></tr>
<tr><td align="left">sentence-transformers</td><td align="left">6.1.0</td></tr>
<tr><td align="left">faiss-cpu</td><td align="left">1.15.1</td></tr>
<tr><td align="left">ModelScope</td><td align="left">1.40.1</td></tr>
<tr><td align="left">FastAPI / Uvicorn</td><td align="left">0.123.10 / 0.54.0</td></tr>
</table>

PyTorch 版本中的 `+cpu` 是当前镜像内预装包的构建标识；NPU 能力由配套的 `torch-npu` 提供，本实践已通过 `torch.npu.is_available()` 和完整链路验证。

运行下面的检查单元后，只有看到绿色通过提示才应继续。如果出现红色提示，应先检查 `npu-smi info`、剩余磁盘空间和当前镜像；该提示本身不会自动终止后续单元。

In [ ]:
import os
import platform
import re
import shutil
import subprocess
from pathlib import Path

# 静默查询 NPU 信息
result = subprocess.run(
    ['timeout', '15s', 'npu-smi', 'info'],
    text=True,
    capture_output=True,
    check=False,
)
output = result.stdout + '\n' + result.stderr

# 读取当前实例的镜像、Python 和 CANN 版本，便于与上方验证快照核对。
image_version_path = Path('/home/developer/image-version.txt')
image_version = image_version_path.read_text(encoding='utf-8').strip() if image_version_path.is_file() else '未知镜像'
cann_home = Path(os.environ.get('ASCEND_HOME_PATH', Path.home() / 'Ascend' / 'cann')).resolve()
cann_version = cann_home.name.removeprefix('cann-') if cann_home.name.startswith('cann-') else '未知版本'
python_version = platform.python_version()

# 解析 NPU 数量、健康状态和硬件型号。
device_rows = re.findall(
    r'^\|\s*(\d+)\s+.*?\|\s*(OK|Warning|Alarm|Abnormal)\s*\|',
    output,
    flags=re.MULTILINE | re.IGNORECASE,
)
visible_devices_text = next((
    os.environ.get(name, '')
    for name in ('ASCEND_VISIBLE_DEVICES', 'NPU_VISIBLE_DEVICES', 'NPU-VISIBLE-DEVICES')
    if os.environ.get(name)
), '')
visible_device_ids = [device_id.strip() for device_id in visible_devices_text.split(',') if device_id.strip()]
npu_count = len(visible_device_ids) or len({device_id for device_id, _ in device_rows})
chip_count = len(device_rows)
health_ok = bool(device_rows) and all(status.upper() == 'OK' for _, status in device_rows)
if re.search(r'910B[1-4]', output, flags=re.IGNORECASE):
    model = '910B'
elif re.search(r'Ascend\s*910', output, flags=re.IGNORECASE):
    model = '910C'
else:
    model = '未知型号'

# 910C 的一张 NPU 可以包含多个芯片；汇总各芯片的 HBM，并统一换算为 GiB。
hbm_totals_gib = []
for _, total, unit in re.findall(r'(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)(?:\s*(MB|MiB|GB|GiB))?', output, flags=re.IGNORECASE):
    total = float(total)
    unit = unit.lower()
    if unit in ('gb', 'gib'):
        hbm_totals_gib.append(total)
    elif unit in ('mb', 'mib', '') and total >= 1024:
        hbm_totals_gib.append(total / 1024)
hbm_total_gib = round(sum(hbm_totals_gib))

# 检查 /mnt/workspace 剩余磁盘空间

# 模型、下载缓存、索引和日志需要较大空间；35 GB 是本实践的最低检查值。
REQUIRED_DISK_GB = 35
free_disk_gb = shutil.disk_usage('/mnt/workspace').free / 1024 ** 3
disk_ok = free_disk_gb >= REQUIRED_DISK_GB

if result.returncode == 0 and health_ok and hbm_total_gib > 0 and disk_ok:
    print(f'🟩 🟩 🟩 环境检查通过，镜像：{image_version}，CANN：{cann_version}，Python：{python_version}，NPU：{model} × {npu_count}（{chip_count}个芯片），HBM共计：{hbm_total_gib} GiB，工作区可用空间：{free_disk_gb:.1f} GB')
else:
    print(f'🟥 🟥 🟥 环境检查未通过，请确认 NPU 状态，且 /mnt/workspace 可用空间不少于 {REQUIRED_DISK_GB} GB（当前 {free_disk_gb:.1f} GB）。')


### 2.1 集中配置工作目录和参数

下面的单元统一定义路径、模型、端口和检索参数。默认工作目录是 `/mnt/workspace/rag_knowledge_base`，重复执行不会删除已有模型、文档或索引。

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">参数</th><th align="left">默认值</th><th align="left">说明</th></tr>
<tr><td align="left"><code>EMBEDDING_DEVICE</code></td><td align="left"><code>npu:0</code></td><td align="left">文档批量向量化所用设备</td></tr>
<tr><td align="left"><code>EMBEDDING_BATCH_SIZE</code></td><td align="left"><code>8</code></td><td align="left">显存不足时可适当调小</td></tr>
<tr><td align="left"><code>LLM_HOST</code> / <code>LLM_PORT</code></td><td align="left"><code>127.0.0.1</code> / <code>8765</code></td><td align="left">本地模型服务地址</td></tr>
<tr><td align="left"><code>RETRIEVAL_CANDIDATE_K</code></td><td align="left"><code>10</code></td><td align="left">向量检索的候选池大小</td></tr>
<tr><td align="left"><code>RETRIEVAL_TOP_K</code></td><td align="left"><code>3</code></td><td align="left">重排后送入生成模型的文档块数</td></tr>
<tr><td align="left"><code>MIN_RETRIEVAL_SCORE</code></td><td align="left"><code>0.40</code></td><td align="left">演示用拒答门槛，需要结合评测校准</td></tr>
</table>

Notebook 中的 `documents`、`index` 和 `embedding_model` 同时保存在当前内核内存中。内核重启后，磁盘上的下载文件和索引不会丢失，但仍需从本节开始依次重建运行时变量；下载单元会复用完整的本地模型文件。

In [ ]:
import os
from pathlib import Path

# 部分 Notebook 内核不会继承完整的 CANN 环境变量；在导入 torch-npu 前补齐 OPP 路径。
ascend_home_path = Path(os.environ.get('ASCEND_HOME_PATH', Path.home() / 'Ascend' / 'cann'))
ascend_opp_path = Path(os.environ.get('ASCEND_OPP_PATH', ascend_home_path / 'opp'))
if not ascend_opp_path.is_dir():
    raise FileNotFoundError(f'未找到 CANN OPP 目录：{ascend_opp_path}，请先加载 CANN set_env.sh。')
os.environ.setdefault('ASCEND_HOME_PATH', str(ascend_home_path))
os.environ.setdefault('ASCEND_OPP_PATH', str(ascend_opp_path))
# 禁用 PyTorch 后端自动加载，后续显式导入 torch_npu，避免 Notebook 重跑时重复注册。
os.environ.setdefault('TORCH_DEVICE_BACKEND_AUTOLOAD', '0')

# 后续步骤共用的配置集中在这里；需要更换目录、设备或检索参数时，只修改本单元。
WORKSPACE = Path('/mnt/workspace') / 'rag_knowledge_base'
MODEL_DIR = WORKSPACE / 'models' / 'llm' / 'Qwen3-8B'
EMBEDDING_MODEL_DIR = WORKSPACE / 'models' / 'embedding' / 'bge-m3'
SOURCE_DIR = WORKSPACE / 'source'
DOCS_DIR = WORKSPACE / 'documents'
INDEX_DIR = WORKSPACE / 'index'
INDEX_PATH = INDEX_DIR / 'cannlab_manual.faiss'
METADATA_PATH = INDEX_DIR / 'cannlab_manual_metadata.jsonl'
CONFIG_DIR = WORKSPACE / 'config'
LOG_DIR = WORKSPACE / 'logs'

EMBEDDING_MODEL_NAME = 'BAAI/bge-m3'
EMBEDDING_MODEL_SIZE_GB = 4.6
EMBEDDING_DEVICE = 'npu:0'
EMBEDDING_BATCH_SIZE = 8
QWEN_MODEL_REPO = 'Qwen/Qwen3-8B'
QWEN_MODEL_SIZE_GB = 16.4
LLM_HOST = '127.0.0.1'
LLM_PORT = 8765
LLM_BASE_URL = f'http://{LLM_HOST}:{LLM_PORT}/v1'
LLM_MODEL = 'Qwen3-8B'
RETRIEVAL_CANDIDATE_K = 10
RETRIEVAL_TOP_K = 3
PROCEDURAL_COMMAND_BONUS = 0.08

# 这是演示初始值，不是通用最优值；应结合第 7 节的样例结果继续校准。
MIN_RETRIEVAL_SCORE = 0.40

for path in (MODEL_DIR, EMBEDDING_MODEL_DIR, SOURCE_DIR, DOCS_DIR, INDEX_DIR, CONFIG_DIR, LOG_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f'🟩 工作目录已准备完成：{WORKSPACE}')


## 3. 导入并处理知识文档

为了提供与 CANNLab 场景直接相关的示例，本实践从 GitCode `cann/infrastructure` 仓库读取 CANNLab 用户手册，并将 Markdown 与图片复制到知识库目录。

### 数据来源记录

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">项目</th><th align="left">内容</th></tr>
<tr><td align="left">仓库</td><td align="left"><code>https://gitcode.com/cann/infrastructure.git</code></td></tr>
<tr><td align="left">文档路径</td><td align="left"><code>docs/CANNLab/V2/CANNLab用户手册.md</code></td></tr>
<tr><td align="left">本次验证分支</td><td align="left"><code>master</code></td></tr>
<tr><td align="left">本次验证提交</td><td align="left"><code>3e186bdbb17a49d1239f97aa3583ddec6e66d672</code></td></tr>
</table>

代码首次克隆时使用远端默认分支，没有强制锁定上述提交。因此该提交是验证快照，不是代码中的固定依赖；远端文档发生变化后，文档块数量和检索结果可能不同。

成功标志：输出导入目录，并能在该目录看到用户手册和 `images` 子目录。已有本地仓库会被复用，不会自动拉取更新。

In [ ]:
import shutil
import subprocess

INFRASTRUCTURE_REPO = SOURCE_DIR / 'infrastructure'
REPOSITORY_URL = 'https://gitcode.com/cann/infrastructure.git'
MANUAL_PATH = Path('docs/CANNLab/V2/CANNLab用户手册.md')

if not INFRASTRUCTURE_REPO.exists():
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(INFRASTRUCTURE_REPO)],
        text=True,
        capture_output=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or 'infrastructure 仓下载失败。')

source_path = INFRASTRUCTURE_REPO / MANUAL_PATH
if not source_path.is_file():
    raise FileNotFoundError(f'未找到用户手册：{MANUAL_PATH}')
shutil.copy2(source_path, DOCS_DIR / source_path.name)
shutil.copytree(source_path.parent / 'images', DOCS_DIR / 'images', dirs_exist_ok=True)

print(f'🟩 🟩 🟩 已导入 CANNLab 用户手册及图片资源：{DOCS_DIR}')


### 3.1 按章节切分并保留来源

固定字符数切分容易将标题与正文割裂。下面的实现先识别 Markdown 标题，再按照约 500 个字符控制文档块大小，并为每个块保存可追溯信息。

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">元数据字段</th><th align="left">含义</th></tr>
<tr><td align="left"><code>source</code></td><td align="left">原始文件名</td></tr>
<tr><td align="left"><code>chunk_id</code></td><td align="left">文件内文档块编号</td></tr>
<tr><td align="left"><code>section</code></td><td align="left">文档块所属标题</td></tr>
<tr><td align="left"><code>start_line</code> / <code>end_line</code></td><td align="left">原文起止行号</td></tr>
<tr><td align="left"><code>content</code></td><td align="left">用于向量化和生成的正文</td></tr>
<tr><td align="left"><code>images</code></td><td align="left">文档块中真实存在的本地图片路径</td></tr>
</table>

成功标志：输出已处理文件数和生成的文档块数，并展示一个同时包含正文与来源字段的样例。只有标题、没有正文的文档块不会进入知识库，避免标题本身产生虚高的检索分数。

#### 换成自己的知识文档

完成示例后，可以用同一套流程构建自己的知识库：

1. 确认第 2.1 节已执行，工作目录配置仍在当前内核中。如果 Qwen 服务仍在运行，先执行第 8 节停止服务，再重新建库。
2. 将 UTF-8 编码、扩展名为 `.md` 或 `.txt` 的文档放入 `DOCS_DIR`，默认是 `/mnt/workspace/rag_knowledge_base/documents/`。当前只扫描该目录顶层，子目录内的文档不会被导入。
3. Markdown 图片按相对路径放置。例如文档中的 `![流程](images/flow.png)` 对应 `documents/images/flow.png`；图片文件需要真实存在，且位于 `documents/` 内。
4. 若只检索自己的资料，将示例手册移到 `documents/` 之外保留；若需要混合检索则保留。此后跳过第 3 节下载导入单元，以免再次加入示例手册。
5. 依次执行本节的文档处理、第 4 节模型准备和第 4.1 节索引构建。已有完整模型会被复用，FAISS 索引和 JSONL 元数据则按当前文档重新写入。随后按第 5 节至第 6.1 节启动服务并进行问答。

第 7 节的固定问题针对 CANNLab 手册。更换知识库后，请在第 6.1 节分别核查有明确依据和无依据的问题，沿用第 7.1 节的证据追溯与拒答原则；WebIDE、NPU 等示例专属答案不作为新知识库的验收项。

> 当前切分器不做语义合并、块间重叠、表格专项处理和文本清洗。对于更复杂的技术文档，应通过固定评测问题判断切分粒度，而不是只凭字符数选择参数。

In [ ]:
import re

IMAGE_PATTERN = re.compile(r'!\[([^\]]*)\]\(([^)]+)\)')
HEADING_PATTERN = re.compile(r'^(#{1,6})\s+(.+?)\s*$')


def extract_local_images(content: str, document_path: Path):
    """只记录知识库目录内真实存在的本地图片，避免越界路径。"""
    images = []
    documents_root = DOCS_DIR.resolve()
    for alt_text, image_reference in IMAGE_PATTERN.findall(content):
        image_reference = image_reference.strip().split(maxsplit=1)[0].strip('<>')
        if image_reference.startswith(('http://', 'https://', 'data:')):
            continue
        image_path = (document_path.parent / image_reference).resolve()
        if image_path.is_relative_to(documents_root) and image_path.is_file():
            images.append({
                'alt': alt_text.strip() or image_path.stem,
                'path': str(image_path.relative_to(documents_root)),
            })
    return images


def split_document(path: Path, chunk_size: int = 500):
    """优先按 Markdown 标题划分，再按长度切块，并保留章节和行号。"""
    lines = path.read_text(encoding='utf-8').splitlines(keepends=True)
    chunks = []
    current_lines = []
    current_size = 0
    current_section = path.stem
    chunk_section = current_section
    start_line = 1

    def flush(end_line: int):
        nonlocal current_lines, current_size, start_line, chunk_section
        content = ''.join(current_lines).strip()
        if content:
            chunks.append({
                'section': chunk_section,
                'start_line': start_line,
                'end_line': end_line,
                'content': content,
                'images': extract_local_images(content, path),
            })
        current_lines = []
        current_size = 0

    for line_number, line in enumerate(lines, start=1):
        heading = HEADING_PATTERN.match(line.strip())
        if heading and current_lines:
            flush(line_number - 1)
        if heading:
            current_section = heading.group(2).strip()
        if not current_lines:
            start_line = line_number
            chunk_section = current_section
        if current_lines and current_size + len(line) > chunk_size:
            flush(line_number - 1)
            start_line = line_number
            chunk_section = current_section
        current_lines.append(line)
        current_size += len(line)

    flush(len(lines))
    return chunks


documents = []
document_paths = sorted([*DOCS_DIR.glob('*.md'), *DOCS_DIR.glob('*.txt')])
for path in document_paths:
    content_chunks = [
        chunk for chunk in split_document(path)
        if any(
            line.strip() and HEADING_PATTERN.match(line.strip()) is None
            for line in chunk['content'].splitlines()
        )
    ]
    for chunk_id, chunk in enumerate(content_chunks):
        documents.append({
            'source': path.name,
            'chunk_id': chunk_id,
            **chunk,
        })

if not documents:
    raise ValueError(f'未在 {DOCS_DIR} 找到 Markdown 或 TXT 文档。')

print(f'🟩 已处理 {len(document_paths)} 个文件，生成 {len(documents)} 个文档块。')
documents[:1]


## 4. 构建本地向量索引

Embedding 模型负责把文档和问题映射到同一个向量空间。本实践使用多语言模型 `BAAI/bge-m3`，首次下载约 4.6 GB；下载完成后会检查配置文件和权重文件是否存在。

模型文件保存在实例工作目录，后续执行不依赖在线 Embedding API。只有首次下载需要外部网络；如果下载被中断，可重新执行该单元继续由 ModelScope 完成本地下载。

成功标志：输出模型实际大小和本地目录。若下载失败，请优先检查 ModelScope 是否安装、实例网络和剩余磁盘空间。

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('modelscope') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'modelscope'], check=True)

import os
import signal
import subprocess
import time

def get_directory_size_bytes(directory: Path) -> int:
    return sum(path.stat().st_size for path in directory.rglob('*') if path.is_file())

def format_remaining_seconds(seconds: float | None) -> str:
    return f'{seconds:.0f} 秒' if seconds is not None else '待计算'

def show_download_status(downloaded_gb: float, speed_mbps: float, remaining_seconds: float | None, elapsed_seconds: float) -> None:
    print(f'{chr(13)}⏳ 下载中：本地目录已写入 {downloaded_gb:.2f} / {EMBEDDING_MODEL_SIZE_GB} GB，当前速度 {speed_mbps:.2f} MB/s，预计剩余 {format_remaining_seconds(remaining_seconds)}，已耗时 {elapsed_seconds:.0f} 秒。', end='', flush=True)

if not ((EMBEDDING_MODEL_DIR / 'config.json').is_file() and (any(EMBEDDING_MODEL_DIR.glob('*.safetensors')) or any(EMBEDDING_MODEL_DIR.glob('*.bin')))):
    download_process = subprocess.Popen(
        ['modelscope', 'download', '--model', EMBEDDING_MODEL_NAME, '--local_dir', str(EMBEDDING_MODEL_DIR)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    total_bytes = EMBEDDING_MODEL_SIZE_GB * 1024 ** 3
    start_time = time.monotonic()
    last_time = start_time
    last_size_bytes = get_directory_size_bytes(EMBEDDING_MODEL_DIR)
    time.sleep(1)
    current_size_bytes = get_directory_size_bytes(EMBEDDING_MODEL_DIR)
    current_time = time.monotonic()
    speed_mbps = max(0.0, (current_size_bytes - last_size_bytes) / 1024 ** 2 / (current_time - last_time))
    remaining_seconds = max(0.0, total_bytes - current_size_bytes) / (speed_mbps * 1024 ** 2) if speed_mbps else None
    downloaded_gb = current_size_bytes / 1024 ** 3
    elapsed_seconds = current_time - start_time
    print(f'⏳ 开始通过 ModelScope 下载 bge-m3（约 {EMBEDDING_MODEL_SIZE_GB} GB）；')
    show_download_status(downloaded_gb, speed_mbps, remaining_seconds, elapsed_seconds)
    last_time = current_time
    last_size_bytes = current_size_bytes
    try:
        while download_process.poll() is None:
            time.sleep(1)
            current_size_bytes = get_directory_size_bytes(EMBEDDING_MODEL_DIR)
            current_time = time.monotonic()
            speed_mbps = max(0.0, (current_size_bytes - last_size_bytes) / 1024 ** 2 / (current_time - last_time))
            remaining_seconds = max(0.0, total_bytes - current_size_bytes) / (speed_mbps * 1024 ** 2) if speed_mbps else None
            downloaded_gb = current_size_bytes / 1024 ** 3
            elapsed_seconds = current_time - start_time
            show_download_status(downloaded_gb, speed_mbps, remaining_seconds, elapsed_seconds)
            last_time = current_time
            last_size_bytes = current_size_bytes
    except KeyboardInterrupt:
        os.killpg(download_process.pid, signal.SIGTERM)
        try:
            download_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(download_process.pid, signal.SIGKILL)
        print()
        print('🟥 🟥 🟥 已中止 bge-m3 下载，下载子进程已结束。')
        raise
    print()
    if download_process.returncode != 0:
        raise RuntimeError('ModelScope bge-m3 模型下载失败。')

has_embedding_config = (EMBEDDING_MODEL_DIR / 'config.json').is_file()
has_embedding_weights = any(EMBEDDING_MODEL_DIR.glob('*.safetensors')) or any(EMBEDDING_MODEL_DIR.glob('*.bin'))
if not (has_embedding_config and has_embedding_weights):
    raise RuntimeError('bge-m3 下载不完整，缺少配置或权重文件。')

model_size_gb = get_directory_size_bytes(EMBEDDING_MODEL_DIR) / 1024 ** 3
print(f'🟩 🟩 🟩 bge-m3 已下载完成，模型大小：{model_size_gb:.2f} GB，目录：{EMBEDDING_MODEL_DIR}')


### 4.1 在 NPU 向量化并持久化

该步骤在 NPU 上批量生成文档向量，归一化后写入 FAISS `IndexFlatIP`，并同步保存 JSONL 元数据。

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">文件</th><th align="left">默认位置</th><th align="left">作用</th></tr>
<tr><td align="left">向量索引</td><td align="left"><code>index/cannlab_manual.faiss</code></td><td align="left">保存文档向量并执行 Top-K 查询</td></tr>
<tr><td align="left">元数据</td><td align="left"><code>index/cannlab_manual_metadata.jsonl</code></td><td align="left">将检索结果映射回正文和来源</td></tr>
</table>

构建结束后，Embedding 模型会移动到 CPU，并清理 NPU 缓存。这样既能继续为单条问题生成向量，又能把 `npu:0` 的显存留给 Qwen3-8B。

成功标志：输出索引中的文档块数量、索引路径以及 NPU 显存已释放的提示。FAISS 文件与 JSONL 必须成对保留；只复制其中一个无法恢复完整知识库。

In [ ]:
import gc
import importlib.util
import json
import subprocess
import sys

required_packages = {
    'sentence_transformers': 'sentence-transformers',
    'faiss': 'faiss-cpu',
    'pandas': 'pandas',
}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing_packages], check=True)

import torch
import torch_npu
from sentence_transformers import SentenceTransformer
import faiss

if not documents:
    raise ValueError('未找到文档块，请先执行第 3.1 节。')
if not torch.npu.is_available():
    raise RuntimeError('未检测到可用 NPU，无法在本地 NPU 运行 bge-m3。')

embedding_model = SentenceTransformer(str(EMBEDDING_MODEL_DIR), device=EMBEDDING_DEVICE)
document_vectors = embedding_model.encode(
    [item['content'] for item in documents],
    batch_size=EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

index = faiss.IndexFlatIP(document_vectors.shape[1])
index.add(document_vectors)
faiss.write_index(index, str(INDEX_PATH))
with METADATA_PATH.open('w', encoding='utf-8') as metadata_file:
    for item in documents:
        metadata_file.write(json.dumps(item, ensure_ascii=False) + '\n')

# 文档批量向量化使用 NPU；随后将模型移至 CPU，给 Qwen3-8B 释放 NPU 显存。
# 后续单条问题向量化继续复用该模型，避免再次占用 NPU。
embedding_model.to('cpu')
gc.collect()
torch.npu.empty_cache()

print(f'🟩 已构建 {index.ntotal} 个文档块的本地索引：{INDEX_PATH}')
print('🟩 Embedding 模型已移至 CPU，NPU 显存已释放给生成模型。')


## 5. 部署本地 Qwen3-8B 服务

生成模型使用 `Qwen/Qwen3-8B`，首次下载预计约 16.4 GB。下载单元会读取权重索引，并逐一确认所需分片是否存在；完整文件会直接复用，中断后可重新执行。

模型只从本地目录加载，推理阶段不向外部服务发送问题、检索证据或答案。模型权重及其许可证仍由模型发布方管理，将本 Notebook 用于其他发布或商业场景前，应单独核对对应模型许可。

成功标志：输出模型实际大小和本地目录。如果始终反复下载，应检查 `config.json`、`model.safetensors.index.json` 和索引中列出的全部权重分片。

In [ ]:
import importlib.util
import json
import os
import signal
import subprocess
import sys
import time

def show_qwen_download_status(downloaded_gb: float, speed_mbps: float, remaining_seconds: float | None, elapsed_seconds: float) -> None:
    print(f'{chr(13)}⏳ 下载中：本地目录已写入 {downloaded_gb:.2f} / {QWEN_MODEL_SIZE_GB} GB，当前速度 {speed_mbps:.2f} MB/s，预计剩余 {format_remaining_seconds(remaining_seconds)}，已耗时 {elapsed_seconds:.0f} 秒。', end='', flush=True)

if importlib.util.find_spec('modelscope') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'modelscope'], check=True)

def get_model_size_bytes(directory: Path) -> int:
    return sum(path.stat().st_size for path in directory.rglob('*') if path.is_file())

def format_remaining_seconds(seconds: float | None) -> str:
    return f'{seconds:.0f} 秒' if seconds is not None else '待计算'

def qwen_model_is_complete() -> bool:
    config_path = MODEL_DIR / 'config.json'
    index_path = MODEL_DIR / 'model.safetensors.index.json'
    if not (config_path.is_file() and index_path.is_file()):
        return False
    try:
        index_data = json.loads(index_path.read_text(encoding='utf-8'))
        expected_weights = set(index_data.get('weight_map', {}).values())
    except (OSError, json.JSONDecodeError):
        return False
    return bool(expected_weights) and all((MODEL_DIR / filename).is_file() for filename in expected_weights)

download_cancelled = False
if not qwen_model_is_complete():
    download_process = subprocess.Popen(
        ['modelscope', 'download', '--model', QWEN_MODEL_REPO, '--local_dir', str(MODEL_DIR)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    total_bytes = QWEN_MODEL_SIZE_GB * 1024 ** 3
    start_time = time.monotonic()
    last_time = start_time
    last_size_bytes = get_model_size_bytes(MODEL_DIR)
    time.sleep(1)
    current_size_bytes = get_model_size_bytes(MODEL_DIR)
    current_time = time.monotonic()
    speed_mbps = max(0.0, (current_size_bytes - last_size_bytes) / 1024 ** 2 / (current_time - last_time))
    remaining_seconds = max(0.0, total_bytes - current_size_bytes) / (speed_mbps * 1024 ** 2) if speed_mbps else None
    downloaded_gb = current_size_bytes / 1024 ** 3
    elapsed_seconds = current_time - start_time
    print(f'⏳ 开始通过 ModelScope 下载 Qwen3-8B（约 {QWEN_MODEL_SIZE_GB} GB），预计需要 40 分钟；')
    show_qwen_download_status(downloaded_gb, speed_mbps, remaining_seconds, elapsed_seconds)
    last_time = current_time
    last_size_bytes = current_size_bytes
    try:
        while download_process.poll() is None:
            time.sleep(1)
            current_size_bytes = get_model_size_bytes(MODEL_DIR)
            current_time = time.monotonic()
            speed_mbps = max(0.0, (current_size_bytes - last_size_bytes) / 1024 ** 2 / (current_time - last_time))
            remaining_seconds = max(0.0, total_bytes - current_size_bytes) / (speed_mbps * 1024 ** 2) if speed_mbps else None
            downloaded_gb = current_size_bytes / 1024 ** 3
            elapsed_seconds = current_time - start_time
            show_qwen_download_status(downloaded_gb, speed_mbps, remaining_seconds, elapsed_seconds)
            last_time = current_time
            last_size_bytes = current_size_bytes
    except KeyboardInterrupt:
        os.killpg(download_process.pid, signal.SIGTERM)
        try:
            download_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(download_process.pid, signal.SIGKILL)
        print()
        print('🟥 🟥 🟥 已中止 Qwen3-8B 下载，下载子进程已结束。')
        download_cancelled = True
    if not download_cancelled:
        print()
        if download_process.returncode != 0:
            raise RuntimeError('Qwen3-8B 模型下载失败。')

if not download_cancelled:
    if not qwen_model_is_complete():
        raise RuntimeError('Qwen3-8B 下载不完整，请重新执行本单元以继续断点续传。')
    model_size_gb = get_model_size_bytes(MODEL_DIR) / 1024 ** 3
    print(f'🟩 🟩 🟩 Qwen3-8B 已下载完成，模型大小：{model_size_gb:.2f} GB，目录：{MODEL_DIR}')


### 5.1 启动简化的聊天接口

下面的单元生成一个最小 FastAPI 服务脚本，在 `npu:0` 加载本地权重，并启动仅监听 `127.0.0.1:8765` 的进程。

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">接口</th><th align="left">用途</th></tr>
<tr><td align="left"><code>GET /v1/models</code></td><td align="left">检查服务状态和已加载模型</td></tr>
<tr><td align="left"><code>POST /v1/chat/completions</code></td><td align="left">接收消息、生成回答</td></tr>
</table>

运行时文件默认位于：

- 服务脚本：`config/qwen3_8b_http_server.py`；
- 服务日志：`logs/qwen3_8b_http_server.log`；
- 进程记录：`config/qwen3_8b_http_server.pid`。

如果端口已经被占用，单元会请求模型列表，确认它确实是预期的 Qwen3-8B 服务；如果端口属于其他程序，会直接报错而不是重复加载模型。

> 该服务没有鉴权、限流、并发队列、流式输出和生产级监控。监听本机地址是当前示例的重要安全边界，不应为了远程访问直接改成公网监听。

In [ ]:
import importlib.util
import os
import socket
import subprocess
import sys
import textwrap

SERVER_FILE = CONFIG_DIR / 'qwen3_8b_http_server.py'
SERVER_LOG = LOG_DIR / 'qwen3_8b_http_server.log'
SERVER_PID_FILE = CONFIG_DIR / 'qwen3_8b_http_server.pid'
if not (MODEL_DIR / 'config.json').is_file():
    raise FileNotFoundError('未找到 Qwen3-8B 模型，请先执行第 5 节下载模型。')

required_packages = {'fastapi': 'fastapi', 'uvicorn': 'uvicorn', 'requests': 'requests'}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing_packages], check=True)

server_source = textwrap.dedent(f'''
    import time
    import torch
    import torch_npu
    import uvicorn
    from fastapi import FastAPI, HTTPException
    from pydantic import BaseModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    MODEL_PATH = {str(MODEL_DIR)!r}
    MODEL_NAME = {LLM_MODEL!r}
    DEVICE = 'npu:0'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16, local_files_only=True).to(DEVICE).eval()
    app = FastAPI()

    class ChatRequest(BaseModel):
        model: str
        messages: list[dict]
        temperature: float = 0.2
        max_tokens: int = 512

    @app.get('/v1/models')
    def list_models():
        return {{'object': 'list', 'data': [{{'id': MODEL_NAME, 'object': 'model'}}]}}

    @app.post('/v1/chat/completions')
    def chat(request: ChatRequest):
        if request.model != MODEL_NAME:
            raise HTTPException(status_code=404, detail=f'未加载模型：{{request.model}}')
        prompt = tokenizer.apply_chat_template(request.messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer([prompt], return_tensors='pt').to(DEVICE)
        with torch.inference_mode():
            generation_args = {{'max_new_tokens': request.max_tokens, 'do_sample': request.temperature > 0}}
            if request.temperature > 0:
                generation_args['temperature'] = request.temperature
            output_ids = model.generate(**inputs, **generation_args)
        answer = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        return {{'id': f'chatcmpl-{{int(time.time())}}', 'object': 'chat.completion', 'model': MODEL_NAME, 'choices': [{{'index': 0, 'message': {{'role': 'assistant', 'content': answer}}, 'finish_reason': 'stop'}}]}}

    uvicorn.run(app, host={LLM_HOST!r}, port={LLM_PORT})
''').strip() + '\n'
SERVER_FILE.write_text(server_source, encoding='utf-8')

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_probe:
    port_in_use = port_probe.connect_ex((LLM_HOST, LLM_PORT)) == 0

if port_in_use:
    import requests
    try:
        response = requests.get(f'{LLM_BASE_URL}/models', timeout=5)
        response.raise_for_status()
        model_ids = [item.get('id') for item in response.json().get('data', [])]
    except Exception as error:
        raise RuntimeError(f'端口 {LLM_PORT} 已占用，但不是预期的模型服务：{error}') from error
    if LLM_MODEL not in model_ids:
        raise RuntimeError(f'端口 {LLM_PORT} 上未加载预期模型 {LLM_MODEL}。')
    server_process = None
    print(f'🟩 本地模型服务已运行：{LLM_BASE_URL}，未重复启动。')
else:
    server_env = os.environ.copy()
    # 保留 CANNLab 分配的可见设备映射；服务内的 npu:0 是映射后的逻辑设备。
    with SERVER_LOG.open('w', encoding='utf-8') as server_log_file:
        server_process = subprocess.Popen(
            [sys.executable, str(SERVER_FILE)],
            stdout=server_log_file,
            stderr=subprocess.STDOUT,
            env=server_env,
            start_new_session=True,
        )
    SERVER_PID_FILE.write_text(str(server_process.pid), encoding='utf-8')
    print(f'⏳ 正在本地 NPU 启动 Qwen3-8B HTTP 服务，进程 PID：{server_process.pid}，日志：{SERVER_LOG}')


### 5.2 等待模型服务就绪

Qwen3-8B 权重加载可能需要数分钟。下面的单元每 2 秒检查一次模型列表，最长等待 10 分钟；看到返回的模型 ID 为 `Qwen3-8B` 后，才表示服务可以进入问答阶段。如果子进程提前退出，会立即展示日志末尾，不再空等到超时。

如果等待超时，按以下顺序定位：

1. 查看 `logs/qwen3_8b_http_server.log` 中最早出现的异常；
2. 用 `npu-smi info` 检查 NPU 状态和剩余显存；
3. 确认 Qwen3-8B 的配置、权重索引和全部分片完整；
4. 确认 `8765` 端口没有被非预期进程占用；
5. 结束失败的服务进程后再重新执行启动单元。

不要通过反复运行启动单元掩盖加载错误；模型、依赖或显存问题应以服务日志为准。

In [ ]:
# 等待本地 NPU 推理服务完成模型加载并开始监听。
import time
import requests

WAIT_TIMEOUT_SECONDS = 600
POLL_INTERVAL_SECONDS = 2
deadline = time.monotonic() + WAIT_TIMEOUT_SECONDS
last_error = None
print('⏳ 等待本地 Qwen3-8B HTTP 服务就绪。')

def read_server_log_tail(max_lines: int = 30) -> str:
    if not SERVER_LOG.is_file():
        return '服务日志尚未生成。'
    return '\n'.join(SERVER_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-max_lines:])

while time.monotonic() < deadline:
    if server_process is not None and server_process.poll() is not None:
        return_code = server_process.wait()
        SERVER_PID_FILE.unlink(missing_ok=True)
        raise RuntimeError(f'模型服务进程提前退出，返回码 {return_code}。日志末尾：\n{read_server_log_tail()}')
    try:
        response = requests.get(f'{LLM_BASE_URL}/models', timeout=10)
        response.raise_for_status()
        model_info = response.json()
        model_ids = [item.get('id') for item in model_info.get('data', [])]
        if LLM_MODEL not in model_ids:
            raise RuntimeError(f'服务已响应，但未加载预期模型 {LLM_MODEL}：{model_ids}')
        print(f'🟩 🟩 🟩 本地 Qwen3-8B HTTP 服务验证通过：{model_info}')
        break
    except requests.RequestException as error:
        last_error = error
        time.sleep(POLL_INTERVAL_SECONDS)
else:
    raise RuntimeError(
        f'等待本地 HTTP 服务 {WAIT_TIMEOUT_SECONDS} 秒后仍未就绪：{last_error}。'
        f'\n日志末尾：\n{read_server_log_tail()}'
    )


## 6. 串起检索与生成

这里将应用逻辑封装为三个层次：

1. `retrieve_documents`：先召回候选池，过滤纯标题和目录块，再对操作型问题中的命令证据做轻量重排；
2. `answer_with_local_llm`：判断证据是否达到门槛，并调用本地模型；
3. `run_rag` / `display_rag_result`：记录耗时并展示回答、来源和相关图片。

检索先从向量索引召回 `RETRIEVAL_CANDIDATE_K` 个候选，再过滤纯标题和目录块。对于“如何、查看、排查”等操作型问题，如果章节名与问题共享技术词且正文包含命令，会增加少量排序分；最终仍只把 `RETRIEVAL_TOP_K` 个证据交给模型。该规则只改变排序，不篡改向量相似度。

当入选证据的最高向量相似度低于 `MIN_RETRIEVAL_SCORE` 时，系统直接拒答，不把弱相关文本交给模型。默认值 `0.40` 只是当前示例的起点，不是 bge-m3 的通用阈值。阈值只能过滤语义相关性较弱的文本，不能证明证据中一定包含答案；阈值过低容易让无关材料进入提示词，阈值过高则可能拒绝本可回答的问题。

提示词要求模型仅依据给定资料回答，并使用 `[1]`、`[2]` 等编号标注关键结论来源。这里的编号对应本次检索结果，不是原始文档的固定编号。

> 知识库内容会进入模型提示词，因此只应导入可信文档。基础 RAG 提示不能自动消除文档中的恶意指令或错误内容。

In [ ]:
from IPython.display import Image as NotebookImage, Markdown, display
import time
import requests


ACTION_QUERY_TERMS = ('如何', '怎么', '怎样', '查看', '检查', '排查', '命令')
TECHNICAL_TOKEN_PATTERN = re.compile(r'[A-Za-z][A-Za-z0-9_.+-]{1,}')
COMMAND_PATTERN = re.compile(r'(?m)^\s*(?:sudo\s+)?[a-z][\w.-]*(?:\s+[-\w|./]+)+\s*$')
DOMAIN_TECHNICAL_TOKENS = {'cannlab'}
INSUFFICIENT_EVIDENCE_ANSWER = '当前知识库没有提供足够依据，无法确认答案。'


def is_navigation_chunk(document: dict) -> bool:
    content_lines = [line.strip() for line in document['content'].splitlines() if line.strip()]
    return document['section'] == '目录' or (bool(content_lines) and all(line.startswith('#') for line in content_lines))


def procedural_command_bonus(question: str, document: dict) -> float:
    if not any(term in question for term in ACTION_QUERY_TERMS):
        return 0.0
    if not COMMAND_PATTERN.search(document['content']):
        return 0.0
    question_tokens = {token.lower() for token in TECHNICAL_TOKEN_PATTERN.findall(question)} - DOMAIN_TECHNICAL_TOKENS
    section_tokens = {token.lower() for token in TECHNICAL_TOKEN_PATTERN.findall(document['section'])}
    return PROCEDURAL_COMMAND_BONUS if question_tokens & section_tokens else 0.0


def retrieve_documents(question: str, top_k: int = RETRIEVAL_TOP_K):
    if not documents or index.ntotal == 0:
        raise ValueError('知识库未就绪，请先完成第 4 节。')
    question_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    candidate_k = min(max(top_k, RETRIEVAL_CANDIDATE_K), index.ntotal)
    scores, document_ids = index.search(question_vector, candidate_k)
    candidates = []
    for score, document_id in zip(scores[0], document_ids[0]):
        if document_id < 0:
            continue
        document = documents[document_id]
        dense_score = float(score)
        candidates.append({
            **document,
            'score': dense_score,
            'rank_score': dense_score + procedural_command_bonus(question, document),
        })
    content_candidates = [item for item in candidates if not is_navigation_chunk(item)]
    if len(content_candidates) < top_k:
        content_candidates.extend(item for item in candidates if is_navigation_chunk(item))
    content_candidates.sort(key=lambda item: item['rank_score'], reverse=True)
    return content_candidates[:top_k]


def answer_with_local_llm(question: str, retrieved_documents: list[dict]):
    best_dense_score = max((item['score'] for item in retrieved_documents), default=-1.0)
    if best_dense_score < MIN_RETRIEVAL_SCORE:
        return INSUFFICIENT_EVIDENCE_ANSWER, False

    context = '\n\n'.join(
        f"[{position}] 来源：{item['source']}，章节：{item['section']}，"
        f"行号：{item['start_line']}-{item['end_line']}\n{item['content']}"
        for position, item in enumerate(retrieved_documents, start=1)
    )
    messages = [
        {
            'role': 'system',
            'content': (
                '仅根据给定资料回答。关键结论后使用 [1]、[2] 等编号标注依据；'
                f'资料不足时必须且只能原样回答“{INSUFFICIENT_EVIDENCE_ANSWER}”，不要添加引用或其他文字。'
            ),
        },
        {'role': 'user', 'content': f'资料：\n{context}\n\n问题：{question}'},
    ]
    response = requests.post(
        f'{LLM_BASE_URL}/chat/completions',
        json={'model': LLM_MODEL, 'messages': messages, 'temperature': 0.0, 'max_tokens': 512},
        timeout=180,
    )
    response.raise_for_status()
    return response.json()['choices'][0]['message']['content'], True


def run_rag(question: str, top_k: int = RETRIEVAL_TOP_K):
    started_at = time.perf_counter()
    retrieved = retrieve_documents(question, top_k)
    answer, used_llm = answer_with_local_llm(question, retrieved)
    return {
        'question': question,
        'answer': answer,
        'retrieved': retrieved,
        'used_llm': used_llm,
        'elapsed_seconds': time.perf_counter() - started_at,
    }


def display_rag_result(result: dict, show_images: bool = True):
    display(Markdown(f"### 问题\n\n{result['question']}\n\n### 回答\n\n{result['answer']}"))
    print(f"耗时：{result['elapsed_seconds']:.2f} 秒；调用生成模型：{'是' if result['used_llm'] else '否'}")
    print('检索证据：')
    for position, item in enumerate(result['retrieved'], start=1):
        rank_note = f" / 排序分 {item['rank_score']:.4f}" if item['rank_score'] != item['score'] else ''
        print(
            f"[{position}] {item['source']} / {item['section']} / "
            f"行 {item['start_line']}-{item['end_line']} / 向量相似度 {item['score']:.4f}{rank_note}"
        )

    if not show_images:
        return
    displayed_paths = set()
    related_images = []
    for item in result['retrieved']:
        for image_info in item.get('images', []):
            if image_info['path'] not in displayed_paths:
                related_images.append((item, image_info))
                displayed_paths.add(image_info['path'])
    if related_images:
        display(Markdown('### 相关图片'))
        for item, image_info in related_images:
            image_path = DOCS_DIR / image_info['path']
            display(Markdown(f"**{image_info['alt']}**  \n来源：{item['source']} / {item['section']}"))
            display(NotebookImage(filename=str(image_path), width=900))


### 6.1 完成一次带来源的问答

示例问题聚焦用户手册中的 WebIDE 故障排查。执行后应同时看到：

- 问题和自然语言回答；
- 是否实际调用了生成模型；
- 总耗时；
- Top-K 证据的文件、章节、行号和相似度；
- 命中文档块关联的本地图片（如果存在）。

回答中的 `[1]`、`[2]` 对应下方证据列表。相似度只表示语义相关程度，不等于答案正确率；相关图片也只表示它与命中文档块存在路径关联，不代表模型理解了图片内容。

体验其他问题时，只修改 `question` 后重新执行本单元即可。调整参数时，请按以下顺序操作：

- 修改 `RETRIEVAL_CANDIDATE_K` 或 `PROCEDURAL_COMMAND_BONUS`：修改并执行第 2.1 节配置单元后，可直接重跑问答或评测。
- 修改 `RETRIEVAL_TOP_K`：修改并执行第 2.1 节配置单元，然后重新执行第 6 节函数定义单元，最后运行本节问答或第 7 节评测。Top-K 默认值在函数定义时确定，仅重跑配置不会改变已有函数的默认值。
- 修改 `MIN_RETRIEVAL_SCORE`：修改并执行第 2.1 节配置单元后，可直接重跑问答或评测；阈值在每次调用时读取。

同时调整多个参数时，请在配置单元中保留全部新值，并记录本次 Top-K 和阈值，方便比较结果。

In [ ]:
question = 'CANNLab WebIDE 报错时，应该如何排查？'
result = run_rag(question)
display_rag_result(result)


## 7. 用三个问题做最小验证

单个成功问题只能证明链路工作，不能证明系统在不同输入下都可靠。这里提供三个冒烟样例：两个在示例手册中有明确依据的问题，以及一个主题相关但知识库没有答案的问题。

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">类型</th><th align="left">问题</th><th align="left">最低预期</th></tr>
<tr><td align="left">明确可回答</td><td align="left">WebIDE 报错时如何排查</td><td align="left">命中故障排查章节，回答带依据</td></tr>
<tr><td align="left">明确可回答</td><td align="left">如何查看 NPU 状态</td><td align="left">命中手册第 2.3 节“验证结果”，给出 <code>npu-smi info</code> 并引用依据</td></tr>
<tr><td align="left">知识库无依据</td><td align="left">CANNLab 的 CEO 是谁</td><td align="left">明确说明资料不足，不编造人名；可以由检索门槛直接拒答，也可以由模型依据资料拒答</td></tr>
</table>

“CANNLab 的 CEO 是谁”与手册共享明确的 CANNLab 主题，因此最高语义相似度可能仍达到约 `0.62`；这只表示主题相关，不表示知识库有 62% 的概率能够回答，更不表示答案正确。是否能够回答仍需检查命中文本是否提供了直接依据。

表格代码中的“预期行为”是简要提示，人工验收以上表为准。在本次记录的示例手册中，NPU 状态查询有明确答案，若被拒答，应检查召回和阈值，不能将其作为通过结果。

### 正确理解“系统决策”列

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">显示值</th><th align="left">实际含义</th><th align="left">如何验收</th></tr>
<tr><td align="left">生成回答</td><td align="left">已调用生成模型；返回文本也可能是“资料不足”</td><td align="left">查看正文，判断是否有依据地回答或合理拒答</td></tr>
<tr><td align="left">拒绝回答</td><td align="left">无检索结果，或最高语义相似度未达到门槛，未调用生成模型</td><td align="left">核对原文是否确实无法回答，排除误拒答</td></tr>
</table>

该列反映调用路径，不是答案正确性或拒答质量的自动判定。知识库无依据问题即使显示“生成回答”，只要正文严格拒答且未编造事实，也可以通过这一项验收。

评测表记录最高语义相似度、系统决策、耗时、回答摘要和针对这三个固定样例的自动检查。第一次调用可能包含模型或运行时预热开销，因此三行耗时不能直接作为严格性能对比。

结果以 `evaluation_table` 展示在当前 Notebook 中，不会自动写出评测文件。运行后保存包含输出的 Notebook，即可保留这张表；表格中的回答摘要仅包含前 120 个字符，摘要不足以判断时，可在第 6.1 节逐题重跑，查看完整回答和检索证据。

这一节的作用是快速暴露明显问题，不是正式准确率报告。自动检查只验证固定证据、命令、引用格式和拒答措辞，不能替代人工核对答案事实与引用是否充分。

In [ ]:
import pandas as pd

# 小规模冒烟评测用于暴露问题，不代表正式精度结论。
evaluation_cases = [
    {'question': 'CANNLab WebIDE 报错时，应该如何排查？', 'expected': '命中连接故障证据并带引用', 'check': 'webide'},
    {'question': 'CANNLab 环境中如何查看 NPU 状态？', 'expected': '回答 npu-smi info 并带引用', 'check': 'npu'},
    {'question': 'CANNLab 的 CEO 是谁？', 'expected': '严格输出资料不足的标准拒答', 'check': 'unsupported'},
]

def has_valid_citations(answer: str, evidence_count: int) -> bool:
    citation_numbers = [int(number) for number in re.findall(r'\[(\d+)\]', answer)]
    return bool(citation_numbers) and all(1 <= number <= evidence_count for number in citation_numbers)


def check_evaluation_result(case: dict, result: dict) -> tuple[bool, str]:
    answer = result['answer'].lower()
    evidence = '\n'.join(item['content'] for item in result['retrieved']).lower()
    citations_ok = has_valid_citations(result['answer'], len(result['retrieved']))
    if case['check'] == 'webide':
        evidence_ok = '连接错误' in evidence or 'vscode无法连接' in evidence
        passed = result['used_llm'] and evidence_ok and citations_ok
        return passed, '需命中连接故障证据、调用模型并包含范围合法的 [n] 引用'
    if case['check'] == 'npu':
        passed = result['used_llm'] and 'npu-smi info' in evidence and 'npu-smi info' in answer and citations_ok
        return passed, '证据和回答均需包含 npu-smi info，并包含范围合法的 [n] 引用'
    passed = result['answer'].strip() == INSUFFICIENT_EVIDENCE_ANSWER
    return passed, f'回答必须严格等于：{INSUFFICIENT_EVIDENCE_ANSWER}'

evaluation_rows = []
for case in evaluation_cases:
    result = run_rag(case['question'])
    top_score = max((item['score'] for item in result['retrieved']), default=None)
    passed, check_note = check_evaluation_result(case, result)
    evaluation_rows.append({
        '问题': case['question'],
        '预期行为': case['expected'],
        '最高语义相似度': round(top_score, 4) if top_score is not None else None,
        '证据章节': '；'.join(item['section'] for item in result['retrieved']),
        '系统决策': '生成回答' if result['used_llm'] else '拒绝回答',
        '自动检查': '通过' if passed else '未通过',
        '检查说明': check_note,
        '耗时（秒）': round(result['elapsed_seconds'], 2),
        '回答摘要': result['answer'][:120],
    })

evaluation_table = pd.DataFrame(evaluation_rows)
evaluation_table


### 7.1 如何解读结果

先判断证据，再判断回答：

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">现象</th><th align="left">优先检查</th><th align="left">可能调整</th></tr>
<tr><td align="left">命中错误章节</td><td align="left">文档切分、问题表达</td><td align="left">切分粒度、<code>RETRIEVAL_TOP_K</code></td></tr>
<tr><td align="left">正确证据被拒绝</td><td align="left">最高语义相似度</td><td align="left">适当降低阈值</td></tr>
<tr><td align="left">回答包含证据不支持的事实</td><td align="left">检索正文、最高语义相似度与提示词</td><td align="left">结合样例校准阈值、完善提示词</td></tr>
<tr><td align="left">回答结论没有引用支持</td><td align="left">检索正文和回答 <code>[n]</code></td><td align="left">优化提示词或更换证据</td></tr>
<tr><td align="left">首次请求明显较慢</td><td align="left">服务日志和预热状态</td><td align="left">区分冷启动与稳定态耗时</td></tr>
</table>

### 本实践的验收标准

- 环境、文档处理、索引构建和模型服务均无错误完成；
- 两个明确可回答问题均命中正确章节，回答中的关键结论有对应证据；NPU 状态查询应给出 `npu-smi info`，拒答不算通过；
- 知识库无依据问题严格输出标准拒答且不编造事实；直接拒答和模型生成的标准拒答均可，不以“系统决策”列单独判定；
- 输出可以追溯到文件、章节和行号；
- 最终能够停止本地服务并释放端口。

如果要比较不同模型、切分方案或阈值，应扩展固定问题集，至少记录“是否应回答、参考答案要点、标准来源章节、实际命中、引用正确性和人工判定”，并在相同环境下重复执行。

## 8. 结束后释放模型服务

确认不再继续问答后再执行下面的停止单元。它位于全部验证步骤之后，避免从上到下运行时提前中断服务。

停止服务会释放 Qwen3-8B 占用的 NPU 显存和 `8765` 端口，但不会删除模型、示例文档、FAISS 索引或 JSONL 元数据。

需要区分磁盘复用与内核恢复：

- **磁盘文件可复用**：重新执行下载单元时，完整模型不会再次下载；
- **内存变量不可直接复用**：Notebook 内核重启后，仍需从集中配置开始依次运行文档处理和索引构建相关单元，恢复 `documents`、`index` 与 `embedding_model`。

停止单元只会终止 PID 文件记录且命令行与本 Notebook 服务脚本一致的进程，避免误杀其他 Python 任务。停止成功后应看到服务 PID；再次问答前，需要重新执行服务启动和就绪检查。

In [ ]:
import os
import signal
import time

def is_owned_server_process(pid: int) -> bool:
    try:
        cmdline = Path(f'/proc/{pid}/cmdline').read_text(encoding='utf-8', errors='ignore').replace('\x00', ' ')
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        return False
    return str(SERVER_FILE) in cmdline

server_pid = None
if SERVER_PID_FILE.is_file():
    try:
        server_pid = int(SERVER_PID_FILE.read_text(encoding='utf-8').strip())
    except ValueError:
        SERVER_PID_FILE.unlink(missing_ok=True)

if server_pid is None or not is_owned_server_process(server_pid):
    SERVER_PID_FILE.unlink(missing_ok=True)
    if server_process is not None and server_process.poll() is not None:
        server_process.wait()
    print('本 Notebook 启动的 Qwen3-8B HTTP 服务未运行。')
else:
    os.killpg(os.getpgid(server_pid), signal.SIGTERM)
    deadline = time.monotonic() + 30
    while is_owned_server_process(server_pid) and time.monotonic() < deadline:
        time.sleep(0.2)
    if is_owned_server_process(server_pid):
        os.killpg(os.getpgid(server_pid), signal.SIGKILL)
    if server_process is not None and server_process.pid == server_pid:
        server_process.wait(timeout=5)
    SERVER_PID_FILE.unlink(missing_ok=True)
    print(f'🟩 已停止本地 Qwen3-8B HTTP 服务，PID：{server_pid}')


## 9. 完成情况与后续边界

完成全部单元后，你已经验证了以下闭环：

1. 检查 CANNLab NPU 与磁盘环境；
2. 导入真实技术文档并保留来源；
3. 按章节切分文本，在 NPU 上生成文档向量；
4. 持久化 FAISS 索引和 JSONL 元数据；
5. 在本地 NPU 启动 Qwen3-8B 服务；
6. 检索证据、生成回答并展示引用和相关图片；
7. 通过三个问题检查有依据的回答与合理拒答；
8. 停止服务并保留可复用文件。

### 默认产物位置

<table style="text-align: left; margin-left: 0; margin-right: auto;">
<tr><th align="left">类型</th><th align="left">路径</th></tr>
<tr><td align="left">工作目录</td><td align="left"><code>/mnt/workspace/rag_knowledge_base</code></td></tr>
<tr><td align="left">示例文档</td><td align="left"><code>documents/</code></td></tr>
<tr><td align="left">Embedding 模型</td><td align="left"><code>models/embedding/bge-m3/</code></td></tr>
<tr><td align="left">生成模型</td><td align="left"><code>models/llm/Qwen3-8B/</code></td></tr>
<tr><td align="left">向量索引与元数据</td><td align="left"><code>index/</code></td></tr>
<tr><td align="left">服务脚本与日志</td><td align="left"><code>config/</code>、<code>logs/</code></td></tr>
</table>

本实践验证的是一条最小但完整的本地 RAG 开发链路。PDF/OCR、重排序、增量更新、内核重启后的直接索引加载、系统化阈值校准、正式评测集、多用户服务和生产安全加固属于后续增强项，不应被视为当前版本已经具备的能力。

[返回 AI 应用开发实践目录](./README.md)
